In [ ]:
import os
import numpy as np

from typing import Tuple, Dict, Any, Optional

import jax
import jax.numpy as jnp

import flax.linen as nn
import optax

from functools import partial

from tqdm.auto import tqdm

from load_beamng_data import (
    format_dataset,
    exponential_moving_average
)

from engine_model import (
    EngineModel,
    create_networks,
    NoiseTerm,
    integrate_path,
    batch_training_loss,
    default_extra_loss_fn,
    load_model_from_config
)

from logging_utils import TrainCheckpoints

from data_utils import (
    split_dataset,
    pick_batch_transitions_as_array,
    remove_irrelevant_fields,
    get_mean_and_scale_fields,
    get_min_max_fields,
    sequential_loader_full_dataset
)
TRAINING_LOGS_DIR = os.getcwd() + "/training_files/"

In [ ]:
# Load the dataset used for training
def criteria_valid_traj(data : Dict[str, np.array]):
    # First the vehicle must be in realistic drive mode (as the mode for control)
    is_realistic = data["driveStatus_isRealisticDrive"] == 1
    # For now let's assume 2WD and ig range mode (0)
    valid = (data["brake"] <= 0.001) & \
            (data["pbrake"] <= 0.0001) & \
            (data["driveStatus_mode4WD"] == 0) & \
            (data["driveStatus_modeRangeBox"] == 0) & \
            (data["rear_wheelspeed"] >= 10) & \
            (data["driveStatus_engineRunning"] == 1) & \
            is_realistic
    # valid  = valid & (np.logical_and(data["time"] >141, data["time"] < 155))
    return valid

def extra_processing(data : Dict[str, np.array]):
    # Let's smooth some of the data
    data["rear_wheelspeed"] = exponential_moving_average(data["rear_wheelspeed"], alpha = 0.25)
    data["engine_speed"] = exponential_moving_average(data["engine_speed"], alpha = 0.8)
    data["boost_pressure"] = exponential_moving_average(data["boost_pressure"], alpha = 0.2)
    # Let's compute angular acceleration
    dt = np.mean(np.diff(data["time"]))
    rear_wr_dot = np.gradient(data["rear_wheelspeed"], dt)
    data["rear_wheelspeed_dot"] = exponential_moving_average(rear_wr_dot, alpha = 0.05)
    rear_engine_dot = np.gradient(data["engine_speed"], dt)
    data["engine_speed_dot"] = exponential_moving_average(rear_engine_dot, alpha = 0.2)
    # Let's also add some inverse gear ratio
    data["inv_gear_ratio"] = 1.0 / (8.516 *data["gearRatio"]) # 8.516 ~ final drive ratio
    data["engine_torque"] = data["engineTorque"]

# Let's pick a dataset
m_runs =[9,]
full_dataset = format_dataset(
    m_runs,
    sensor_type = "gtstate",
    vehicle_model = "utv_wild",
    valid_trajectory_cond = criteria_valid_traj,
    extra_processing = extra_processing,
    min_traj_len = 200,
)

In [ ]:
# Get min and max values
min_vals, max_vals = get_min_max_fields(full_dataset)
# Get scale and mean values
mean_vals, scale_vals = get_mean_and_scale_fields(full_dataset)

In [ ]:
# Let's do some data splitting
_no_split = True
_test_ratio = 0.1
_seed = 42
np.random.seed(_seed)

useful_fields = [
    "engine_speed", "boost_pressure", "throttle", "rear_wheelspeed", "time",
    "inv_gear_ratio", "engine_torque"
]
# Remove the irrelevant fields from the dataset
_full_dataset = remove_irrelevant_fields(
    full_dataset, useful_fields
)
if _no_split:
    test_data, train_data = _full_dataset, _full_dataset
else:
    train_data, test_data = split_dataset(_full_dataset, _test_ratio, _seed)

min_vals_useful, max_vals_useful = get_min_max_fields(_full_dataset)
mean_vals_useful, scale_vals_useful = get_mean_and_scale_fields(train_data)

In [ ]:
class EngineModelV2(EngineModel):
    """A model for the engine.
    """
    @property
    def name_latents(self) -> Tuple[str]:
        return ["clutch_state",]
    @property
    def bounds_latents(self) -> Tuple[Dict[str, float]]:
        dict_min = {
            "clutch_state": 0,
        }
        dict_max = {
            "clutch_state": 1.0,
        }
        return dict_min, dict_max

    def setup(self):
        """Setup the model.
        """
        self._boost_dot = create_networks(self.nns["boost_dot"], 1)
        self.boost_dot = lambda x : self._boost_dot(self.with_stop_gradient(x))
        self._engine_torque = create_networks(self.nns["engine_torque"], 1)
        self.engine_torque = lambda x : self._engine_torque(self.with_stop_gradient(x))
        self._clutch_dyn = create_networks(self.nns["clutch_dyn"], 1)
        self.clutch_dyn = lambda x : self._clutch_dyn(self.with_stop_gradient(x))
        self._torque_shaft = create_networks(self.nns["torque_shaft"], 1)
        self.torque_shaft = lambda x : self._torque_shaft(self.with_stop_gradient(x))
        self.tau_gear = self.param("tau_gear", nn.initializers.normal(stddev=1), ())

    def calculate_engine_torque(self, data):
        """Calculate the engine torque.
        """
        features = self.nns["engine_torque"].get(
            "features",
            ["engine_speed", "throttle_unscaled", "boost_pressure"]
        )
        nn_input = self.convert_features_to_nn_input(features, data)
        return self.engine_torque(nn_input)[0]

    def calculate_shaft_torque(self, data):
        """Calculate the shaft torque.
        """
        eng_torque = None
        if "engine_torque" in data:
            eng_torque = data["engine_torque"]
        else:
            eng_torque = self.calculate_engine_torque(data)
        w_e = data["engine_speed"]
        w_r = data["rear_wheelspeed"]
        feats = jnp.array([w_e, w_r, eng_torque])
        return self.torque_shaft(feats)[0]

    def calculate_latent_space_field(self, data):
        """Calculate the latent space field.
        """
        clutch_state = data["clutch_state"]
        feat_gr = self.nns["clutch_dyn"].get(
            "features", ["engine_speed", "torque_shaft"]
        )
        clutch_state_eq = self.clutch_dyn(self.convert_features_to_nn_input(feat_gr, data))[0]
        clutch_state_eq = jax.nn.sigmoid(clutch_state_eq)
        tau_gear = jnp.exp(self.tau_gear)
        clutch_state_dot = (clutch_state_eq - clutch_state) * tau_gear
        return jnp.array([clutch_state_dot,]), {"clutch_eq": clutch_state_eq}

    def calculate_initial_guess_for_latent(self, state, control, times):
        """Calculate the initial guess for the latent space.
        """
        if self.is_initializing():
            return jnp.ones(self.num_latent)
        init_latent_state = jnp.array([1.0,])
        dt = jnp.diff(times)
        def opt_step(latent_state, aux):
            _state, _control, _dt = aux
            _data = self.get_data_from_state_and_control(_state, _control)
            _data ={**_data, **{_n : _v for _n, _v in zip(self.name_latents, latent_state)}}
            _data = {**_data, "torque_shaft": self.calculate_shaft_torque(_data)}
            latent_dot, _ = self.calculate_latent_space_field(_data)
            next_latent_state = latent_state + latent_dot * _dt
            next_latent_state = self.constrain_latent(next_latent_state)
            return next_latent_state, None
        # Now let's integrate the latent space
        final_latent, _ = jax.lax.scan(opt_step, init_latent_state, (state[:-1], control, dt))
        return final_latent

    def vector_field(self, state: jnp.ndarray, control: jnp.ndarray):
        """Compute the vector field.
        """
        if self.is_initializing():
            latent_init = self.calculate_initial_guess_for_latent(state, control, None)
            state = self.combine_state_and_latent(state, latent_init)
        # First, extract the latent state if it exists.
        state, latent_state = self.split_state_latent_var(state)
        data = self.get_data_from_state_and_control(state, control)
        data = {**data, **{_n : _v for _n, _v in zip(self.name_latents, latent_state)}}
        # Start with the engine torque, boost pressure and rear wheel speed
        T_e = self.calculate_engine_torque(data)
        we_u, wr_u = data["engine_speed_unscaled"], data["rear_wheelspeed_unscaled"]
        clutch_state = data["clutch_state"]
        inv_gear_ratio = (wr_u / we_u)
        data = {**data, "engine_torque": T_e, "inv_gear_ratio": inv_gear_ratio}
        dot_pb = self.calculate_boost_pressure(data)
        wr_res = self.calculate_res_wheel_speed(data)
        # Shaft torque
        T_s = self.calculate_shaft_torque(data)
        data = {**data, "torque_shaft": T_s}
        # Wheel speed dynamics
        moment_coeffs = self.get_parameter_by_name("JeJr_1")
        gear_ratio =  clutch_state / (inv_gear_ratio + 1.0e-4)
        dot_wr = gear_ratio * T_s * moment_coeffs + wr_res
        dot_we = T_e - T_s
        state_dot = jnp.array([dot_we, dot_pb, dot_wr])
        # Now calculate the latent space dynamics
        latent_dot, _extra_gear = self.calculate_latent_space_field(data)
        full_state_dot = self.combine_state_and_latent(state_dot, latent_dot)
        extra_return = {
            "engine_torque_est": T_e,
            "torque_shaft": T_s,
            "rear_wheel_speed_res": wr_res,
            "inv_gear_ratio": inv_gear_ratio,
            **_extra_gear
        }
        return full_state_dot, extra_return

In [ ]:
# Let's define and load the model
model_config = {
    "noise": {
        "args": {
            "ignore_noise": False,
            "is_constant_noise": False,
            "stop_gradient": True,
            "max_noise": np.array([1, 0.5, 0.5]),
            "noise_nn": {
                "type": "MLP",
                "args": {
                    "layers_archictecture": [32, 32],
                    "activation_fn": "tanh",
                    "initial_value_range": 0.1,
                    "fake_normalization": False,
                }
            },
        },
    },
    "drift":{
        "args":{
            "_history_size": 10,
            "stop_gradient": True,
            "state_max": max_vals_useful,
            "state_min": min_vals_useful,
            "state_action_mean": mean_vals_useful,
            "state_action_std": scale_vals_useful,
            "nns": {
                "boost_dot": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": [64, 64],
                        "activation_fn": "tanh",
                        "initial_value_range": 0.1,
                        "fake_normalization": False,
                    }
                },
                "engine_torque": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": [64, 64],
                        "activation_fn": "tanh",
                        "initial_value_range": 0.1,
                        "fake_normalization": False,
                    }
                },
                "torque_shaft": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": [32, 32],
                        "activation_fn": "tanh",
                        "initial_value_range": 0.1,
                        "fake_normalization": False,
                    }
                },
                "clutch_dyn": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": [32, 32],
                        "activation_fn": "tanh",
                        "initial_value_range": 0.01,
                        "fake_normalization": False,
                    }
                },
            }
        }
    }
}

In [ ]:
# Load the model
seed = 42
rng_key = jax.random.PRNGKey(seed)
drift, noise, my_params = load_model_from_config(
    model_config, seed=seed, class_drift=EngineModelV2, class_noise=NoiseTerm
)

In [ ]:
drift.name_latents

In [ ]:
import matplotlib
matplotlib.use("Qt5Agg")  # Or another interactive backend like "TkAgg", Qt5Agg
# matplotlib.use("widget")  # Or another interactive backend like "TkAgg", Qt5Agg
import matplotlib.pyplot as plt

In [ ]:
# Some plotting utils
LABELS_TO_PLOT = drift.name_states + \
    ["inv_gear_ratio", "engine_torque", "engine_torque_est", "torque_shaft", "rear_wheel_speed_res"]
max_num_rows = 3
# Optimize for more or less equal number of rows and columns
num_rows = min(max_num_rows, len(LABELS_TO_PLOT))
num_cols = int(np.ceil(len(LABELS_TO_PLOT) / num_rows))
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8), sharex=True)
ax_flatten = axes.flatten()
# Create the lines for the ground truth
lines_gt = {}
for idx, name in enumerate(LABELS_TO_PLOT):
    lines_gt[name], = ax_flatten[idx].plot([], [], label="Ground Truth", color="blue", linewidth=1)
    ax_flatten[idx].set_title(name)
    ax_flatten[idx].grid()
    ax_flatten[idx].set_xlabel("Time [s]")
    ax_flatten[idx].legend()

# List to store multiple preds
pred_lines = { name : [] for name in LABELS_TO_PLOT }

fig.tight_layout()
fig.show()

def update_plots(
    times,
    gt_state,
    pred_state,
    alpha = 0.8,
    do_not_erase= False
):
    for idx, name in enumerate(LABELS_TO_PLOT):
        curr_ax = ax_flatten[idx]
        # Update the ground truth
        if not do_not_erase:
            if name in gt_state:
                lines_gt[name].set_data(times, gt_state[name])
            if name in pred_state:
                for line in pred_lines[name]:
                    line.remove()
                pred_lines[name].clear()
        elif name in gt_state:
            # We want to add te new data to the existing one
            lines_gt[name].set_data(
                np.concatenate((lines_gt[name].get_xdata(), times)),
                np.concatenate((lines_gt[name].get_ydata(), gt_state[name]))
            )
        if name not in pred_state:
            curr_ax.relim()
            curr_ax.autoscale_view()
            continue
        # Create the new prediction line
        n_trajectories = pred_state[name].shape[0]
        for i in range(n_trajectories):
            pred_line, = curr_ax.plot(
                times, pred_state[name][i], color="red", alpha=alpha, linewidth=1,
                label="Prediction" if i == 0 else "_nolegend_"
            )
            pred_lines[name].append(pred_line)
        # Refres axis and so on
        curr_ax.relim()
        curr_ax.autoscale_view()
    fig.canvas.draw()
    fig.canvas.flush_events()

In [ ]:
# Problem configuration
# Problem config for data extraction
problem_config_for_sampling ={
    "names_states": list(drift.name_states) + ["time",],
    "names_controls": list(drift.name_controls),
    "time_dependent_parameters": ["inv_gear_ratio", "engine_torque"],
    "system_physical_params": {},
}
action_sampling_strategy ={
    "default": "first"
}

# Let's define function for prediction to use in plotting
@partial(jax.jit, static_argnames=["num_samples", "num_iter_between_dt"])
def prediction_paths(
    init_state,
    controls,
    times,
    rng_key,
    hist_states,
    params,
    num_samples = 4,
    num_iter_between_dt = 1
):
    # Initialize the state and control
    rng_key = jax.random.split(rng_key, num_samples)
    vmap_integrate = jax.vmap(
        integrate_path,
        in_axes=(None, None, None, 0, None, None, None, None, None, None)
    )
    # Integrate the dynamics for each sample
    paths, extras = vmap_integrate(
        init_state, controls, times, rng_key, params, hist_states, drift, noise,
        True, num_iter_between_dt
    )
    state_dict = { name : paths[..., idx] \
        for idx, name in enumerate(drift.name_states + drift.name_latents)
    }
    return {**extras, **state_dict}

def generate_examples(
    m_params, dataset, rng_key, num_samples,
    stepsize_range=(2, 2), horizon=100, num_iter_between_dt=1,
):
    # Get the batch
    test_batch_data = pick_batch_transitions_as_array(
        dataset, 1, stepsize_range, horizon, 
        problem_config_for_sampling, action_sampling_strategy
    ) # Smple a batch with a single element
    # Extract the state and control
    states, controls, gt_extra = test_batch_data
    states, controls = states[0], controls[0]
    gt_extra = { k : v[0] for k, v in gt_extra.items() }
    _time = states[:, -1]
    states = states[:, :-1]
    # Let's remove the history
    (hist_state, hist_control), (states, controls) = \
        drift.split_state_with_history(states, controls)
    _time_hist = _time[:hist_state.shape[0]]
    _time = _time[-states.shape[0]:]
    gt_extra = {k : v[hist_control.shape[0]:] for k, v in gt_extra.items()}
    hist_states = (hist_state, hist_control, _time_hist)
    # Get the initial state
    preds = prediction_paths(
        states[0], controls, _time, rng_key, hist_states, 
        m_params, num_samples, num_iter_between_dt
    )
    # Convert the groundtruth in a dict
    _time = _time[:-1]
    gt_data = {
        **gt_extra,
        **{name : states[...,:_time.shape[0],idx] for idx, name in enumerate(drift.name_states)},
    }
    pred_data = { name : np.array(v)[...,:_time.shape[0]] for name, v in preds.items() }
    gt_data = { name : np.array(v) for name, v in gt_data.items() }
    return _time, gt_data, pred_data

In [ ]:
# Let's try some plotting
plot_num_samples = 4
plot_dataset = train_data
rng_key, plot_rng_key_test = jax.random.split(rng_key)
plot_stepsize_range = (2,4)
plot_horizon = 100
plot_freq = 40
_num_iter_between_dt = 4

plot_time, plot_gt_state, plot_pred_state = generate_examples(
    my_params, plot_dataset, plot_rng_key_test, plot_num_samples,
    stepsize_range = plot_stepsize_range, horizon = plot_horizon,
    num_iter_between_dt = _num_iter_between_dt
)

# Plot the first example
update_plots(plot_time, plot_gt_state, plot_pred_state, alpha=0.3)

In [ ]:
def map_gradient_update_params(
    params : Dict[str, Any], opt_state : Any, state : jnp.ndarray,
    control : jnp.ndarray, rng_key : jnp.ndarray,
    loss_fn : callable, opt : optax.GradientTransformation
):
    rng_key = jax.random.split(rng_key, state.shape[0])
    grads, featvals = jax.grad(loss_fn, has_aux=True)(params, state, control, rng_key)
    updates, opt_state = opt.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, featvals

def build_optimizer(opt_config : Dict[str, Any]) -> optax.GradientTransformation:
    chain_list = []
    for elem in opt_config:
        name_elem = elem['name']
        m_fn = getattr(optax, name_elem)
        m_params = elem.get('params', {})
        print(f'Function : {name_elem} | params : {m_params}')
        if elem.get('scheduler', False):
            m_params = m_fn(**m_params)
            chain_list.append(optax.scale_by_schedule(m_params))
        else:
            chain_list.append(m_fn(**m_params))
    return optax.chain(*chain_list)

In [ ]:
# Let's setup the training parameters
# Create the optimizer
opt_config = [
    { "name": "scale_by_adam",},
    {
        "name": "linear_schedule",
        "scheduler": True,
        # Whatever scheduler you want. Check optax for documentation
        "params": {
            "init_value": -0.001,
            "end_value": -0.001,
            "transition_steps": 20000
        }
    }
]
opt = build_optimizer(opt_config)
opt_state = opt.init(my_params)


# Setup some training parameters
stepsize_range =[2, 4]
horizon = 100
stepsize_range_test = [2, 4]
horizon_test = 100

# Add the history size
horizon = horizon + drift.history_size
horizon_test = horizon_test + drift.history_size

# Problem config for data extraction and batch sampling
problem_config_for_dataset_extraction ={
    "names_states": list(drift.name_states) + ["time",],
    "names_controls": list(drift.name_controls),
    "time_dependent_parameters": ["rear_wheelspeed",], # Useless here
    "system_physical_params": {},
}
action_sampling_strategy ={
    "default": "first"
}

# Let's define the training loop
train_batch = 32 # Number of sequence of trajectories in a batch
test_batch = 32 # Number of sequence of trajectories in a batch
num_epochs = 500 # Number of epochs
num_batches = 500 # Number of batches iteration per epoch
num_test_batches = 30
_test_freq = 1 # Frequence in terms of number of batches for testing the model
_save_freq = 1 # Frequency in terms of number of batches for saving the model
do_iterate_through_all = False # Use the full batch for training

if do_iterate_through_all:
    load_batch, traj_indexes, num_batches = sequential_loader_full_dataset(
        train_data, train_batch, stepsize_range, horizon,
        problem_config_for_dataset_extraction, action_sampling_strategy
    )

test_freq = int(num_batches * _test_freq)
save_freq = int(num_batches * _save_freq)

# Set up the loss function
reg_loss_args = {
    "pred":{
      "discount": 0.99,
      "discount_dt": np.mean(np.diff(train_data["trajectories"][0]["time"])),
      "num_samples": 4,
      "num_iter_between_dt": 1,
    #   "scaling_factor": 1.0 / jnp.array([10, 20.0, 100]).reshape(1,1,-1)
    },
    "reg_params": {
        "nll_mse": 1,
        "nll_noise": 1,
        "loss_reg": 1.0e-8,
        "loss_noise_max": 1.0e-5,
        # "pen_gear_ratio": 10.0,
    }
}

def _extra_loss(
    params : Dict[str, Any],
    gt_state : jnp.ndarray, # [batch, time, state_dim]
    pred_state : jnp.ndarray,
    extras : Dict[str, Any],
):
    dict_loss = default_extra_loss_fn(params, gt_state, pred_state, extras)
    # Let's add a term penalizing the gear ratio
    inv_gear_ratio = extras["inv_gear_ratio"] # [batch, particle, time]
    gt_we = gt_state[..., drift.name_states.index("engine_speed")]
    gt_wr = gt_state[..., drift.name_states.index("rear_wheelspeed")]
    # Let's add a dimension to gt_* where the num particles is 1
    gt_we = gt_we.reshape(gt_we.shape[0], 1, gt_we.shape[1])[...,:gt_we.shape[1]-1]
    gt_wr = gt_wr.reshape(gt_wr.shape[0], 1, gt_wr.shape[1])[...,:gt_wr.shape[1]-1]
    diff_tern = inv_gear_ratio * gt_we - gt_wr
    diff_tern = diff_tern ** 2
    loss_gear_ratio = jnp.mean( jnp.mean(jnp.sum(diff_tern, axis=-1), axis=-1) )
    return {**dict_loss, "pen_gear_ratio": loss_gear_ratio}

# Create the loss function and jit it. Strip it down to the bare minimum for now
jit_model_loss = jax.jit(
    lambda params, state, control, rng_key : \
    batch_training_loss(params, state, control, rng_key, reg_loss_args, drift, noise, _extra_loss)
)

# Let's jit the map gradient function
jit_gradient_update = jax.jit(
    lambda params, opt_state, state, control, rng_key : \
    map_gradient_update_params(params, opt_state, state, control, rng_key, jit_model_loss, opt)
)

def evaluation_metrics(
    params : Dict[str, Any],
    key : jnp.ndarray,
    test_data : Dict[str, Any]
) -> Dict[str, float]:
    res = {}
    for n_iter in tqdm(range(num_test_batches), leave=False):
        test_batch_data = pick_batch_transitions_as_array(
            test_data, test_batch, stepsize_range_test, horizon_test, 
            problem_config_for_dataset_extraction, action_sampling_strategy
        )
        key, key_loss = jax.random.split(key)
        _, losses = jit_model_loss(params, *test_batch_data[:2], key_loss)
        if len(res) == 0:
            res = {k: np.zeros(num_test_batches) for k in losses.keys()}
        for _key, v in losses.items():
            res[_key][n_iter] = v
    return {k: np.mean(v) for k, v in res.items()}


In [ ]:
# Now let's prepare for the training.
save_name = "test_10"

# Let's create the checkpoint manager
track_n_checkoints = {
    "max_to_keep": 5,
    "async_exec": False,
    "metrics": {
        "Train/mse": 1.0,
        "Test/mse": 2.0
    }
}

# First let's create the checkpoint manager
ckpt_model = TrainCheckpoints(
    TRAINING_LOGS_DIR,
    save_name,
    track_n_checkoints,
    best_mode = 'min',
    writer_on = True,
    extra_config_to_save_as_yaml=model_config,
    saving_freq=save_freq
)

# Evaluate through the number of epochs
grad_step = 0
_params = my_params
for _ in tqdm(range(num_epochs)):
    # Shuffle the data indexes
    if do_iterate_through_all:
        np.random.shuffle(traj_indexes)
    
    for n_batch in tqdm(range(num_batches), leave=False):
        if do_iterate_through_all:
            train_batch_data = load_batch(traj_indexes, n_batch)
        else:
            train_batch_data = pick_batch_transitions_as_array(
                train_data, train_batch, stepsize_range, horizon, 
                problem_config_for_dataset_extraction, action_sampling_strategy
            )
            
        # Check if it's time for testing
        if grad_step % test_freq == 0:
            rng_key, rng_test_key = jax.random.split(rng_key)
            eval_metrics = evaluation_metrics(_params, rng_test_key, test_data)
            eval_metrics = {
                f"Test/{k}" : v for k, v in eval_metrics.items()
            }
        # Apply gradient updates
        rng_key, rng_grad_key = jax.random.split(rng_key)
        _params, opt_state, train_metrics = jit_gradient_update(
            _params, opt_state, *train_batch_data[:2], rng_grad_key
        )
        train_metrics = {
            f"Train/{k}" : v for k, v in train_metrics.items()
        }
        grad_step += 1

        # Let's plot if needed
        if grad_step % plot_freq == 0:
            rng_key, plot_rng_key = jax.random.split(rng_key)
            plot_time, plot_gt_state, plot_pred_state = generate_examples(
                _params, plot_dataset, plot_rng_key_test, plot_num_samples,
                stepsize_range = plot_stepsize_range, horizon = plot_horizon,
                num_iter_between_dt = _num_iter_between_dt
            )
            # Plot the first example
            update_plots(plot_time, plot_gt_state, plot_pred_state, alpha=0.3)

        # Save metrics
        metrics_save = {**train_metrics, **eval_metrics}
        save_dict = {
            "sde_params": _params,
            "train_metrics": train_metrics,
            "test_metrics": eval_metrics,
            "current_step": grad_step
        }
        # Write the checkpoint
        ckpt_model.write_checkpoint_and_log_data(save_dict, metrics_save)